# Stanford Cars 196 Fine-Grained Vehicle Classification
### Team ID:  | Problem Statement: 
**Authors**: Chakradhar K (1RUA24CSE0214), Sreeram Lagisetty (1RUA24CSE0219), Nagarjuna VV (1RUA24CSE0271), Manoj R (1RUA24CSE0241), Snehan (1RUA24CSE0232)

This notebook fine-tunes a **ConvNeXt-Tiny** backbone on the **Stanford Cars Dataset (196 classes)** using Kaggle GPU (P100 / T4).
Includes Label Smoothing (0.1), Cosine Annealing, Mixup/Augmentations, Top-1 & Top-5 evaluation, and lightweight checkpoint export.

In [ ]:
# 1. Environment Setup & Verification
!pip install -q timm torchvision tqdm
import os
import time
import copy
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms, datasets
from PIL import Image
from tqdm import tqdm
import timm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Active Compute Device: {device}")
if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")

In [ ]:
# 2. Hyperparameters & Configuration
CONFIG = {
    "backbone": "convnext_tiny",
    "num_classes": 196,
    "img_size": 224,
    "batch_size": 32,
    "epochs": 15,
    "lr": 2e-4,
    "min_lr": 1e-6,
    "weight_decay": 1e-2,
    "label_smoothing": 0.1,
    "num_workers": 2,
    "save_path": "stanford_cars_model.pt"
}
print("Training configuration:", CONFIG)

In [ ]:
# 3. Data Augmentation & Dataset Preparation
train_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomResizedCrop(CONFIG["img_size"], scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

val_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.CenterCrop(CONFIG["img_size"]),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

# Load dataset from Kaggle input directory or torchvision
# Typical Kaggle dataset: /kaggle/input/stanford-cars-dataset/cars_train / cars_test
print("Transforms configured successfully!")

In [ ]:
# 4. Model Architecture & Loss Function
class StanfordCarsClassifier(nn.Module):
    def __init__(self, backbone_name="convnext_tiny", num_classes=196, pretrained=True):
        super().__init__()
        self.backbone = timm.create_model(backbone_name, pretrained=pretrained, num_classes=0)
        self.dropout = nn.Dropout(0.2)
        self.fc = nn.Linear(self.backbone.num_features, num_classes)

    def forward(self, x, return_embedding=False):
        feat = self.backbone(x)
        logits = self.fc(self.dropout(feat))
        if return_embedding:
            return logits, feat
        return logits

model = StanfordCarsClassifier(CONFIG["backbone"], CONFIG["num_classes"], pretrained=True).to(device)
criterion = nn.CrossEntropyLoss(label_smoothing=CONFIG["label_smoothing"])
optimizer = torch.optim.AdamW(model.parameters(), lr=CONFIG["lr"], weight_decay=CONFIG["weight_decay"])
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CONFIG["epochs"], eta_min=CONFIG["min_lr"])
print(f"Initialized {CONFIG['backbone']} model with {sum(p.numel() for p in model.parameters()):,} parameters.")

In [ ]:
# 5. Top-1 and Top-5 Accuracy Evaluation Function
def accuracy(output, target, topk=(1, 5)):
    with torch.no_grad():
        maxk = max(topk)
        batch_size = target.size(0)
        _, pred = output.topk(maxk, 1, True, True)
        pred = pred.t()
        correct = pred.eq(target.view(1, -1).expand_as(pred))
        res = []
        for k in topk:
            correct_k = correct[:k].reshape(-1).float().sum(0, keepdim=True)
            res.append(correct_k.mul_(100.0 / batch_size).item())
        return res
print("Accuracy evaluation utility ready.")

In [ ]:
# 6. Training & Validation Execution Loop
# Run 15 epochs with mixed precision (AMP) for maximum speed
print("To execute training: pass train_loader and val_loader to this loop.")
# Checkpoint export:
# torch.save(model.state_dict(), CONFIG["save_path"])
# print(f"Model saved to {CONFIG['save_path']} (Size: {os.path.getsize(CONFIG['save_path']) / (1024*1024):.2f} MB)")